In [3]:
import random
import numpy as np

# --- USER CONSTRAINT GATHERING ---
print("="*55)
print("  PSO PERSONALIZED FITNESS ROUTINE OPTIMIZER")
print("="*55)
user_age = float(input("Enter user age (e.g., 35): "))
max_days = int(input("Maximum workout days allowed per week? (1-7): "))
required_rest_days = 7 - max(1, min(7, max_days))
age_scalar = max(1.0, user_age / 30.0)

# Realistic variable boundaries [Min, Max]
BOUNDS = [
    (0.0, 6.0),    # Cardio hours/week
    (0.0, 6.0),    # Strength training hours/week
    (0.5, 4.0),    # Mobility hours/week
    (-600, 300),   # Daily calorie delta (deficit/surplus)
    (required_rest_days, 4.0) # Rest days/week
]

# --- STEP 4: FITNESS EVALUATION FUNCTION ---
def evaluate_fitness(position):
    cardio, strength, mobility, calories, rest = position

    # 1. Physiological Adaptation (Maximize)
    adaptation = 0.0
    ideal_strength = 2.5 * age_scalar
    adaptation += (40.0 + (strength - ideal_strength) * 2) if strength >= ideal_strength else (strength / ideal_strength) * 40.0
    adaptation += min(cardio, 3.0) * 10
    adaptation += min(mobility, 2.0) * 15

    # 2. Systemic Stress & Cortisol Management (Minimize)
    stress = 10.0
    total_hours = cardio + strength + mobility
    workout_days = 7 - rest

    if workout_days > 0 and (total_hours / workout_days) > 2.0:
        stress += ((total_hours / workout_days) - 2.0) * 20 * age_scalar
    if rest < required_rest_days:
        stress += (required_rest_days - rest) * 30
    if rest < 1:
        stress += 50  # Hard biological crash penalty for no rest

    # Aggregate fitness score (High adaptation, Low stress)
    fitness_score = adaptation - stress
    return max(fitness_score, 0.1)

# --- PARTICLE CLASS DESIGN ---
class Particle:
    def __init__(self):
        # Step 3: Initialize random positions and velocities
        self.position = np.array([random.uniform(b[0], b[1]) for b in BOUNDS])
        self.velocity = np.array([random.uniform(-1, 1) for _ in BOUNDS])
        self.pbest_position = np.copy(self.position)
        self.pbest_fitness = evaluate_fitness(self.position)

    def update_position(self):
        self.position = self.position + self.velocity
        # Clip positions so particles do not fly outside realistic human capabilities
        for i in range(len(BOUNDS)):
            self.position[i] = np.clip(self.position[i], BOUNDS[i][0], BOUNDS[i][1])

# --- PSO OPTIMIZATION RUNTIME ---
def run_pso(swarm_size=40, max_iterations=80):
    # Step 2: Set PSO Parameters
    w = 0.5   # Inertia weight
    c1 = 1.5  # Cognitive coefficient (personal best pull)
    c2 = 1.5  # Social coefficient (global best pull)

    # Step 3: Initialize Particles Swarm
    swarm = [Particle() for _ in range(swarm_size)]

    # Initialize Global Best
    gbest_position = np.copy(swarm[0].position)
    gbest_fitness = swarm[0].pbest_fitness

    for p in swarm:
        if p.pbest_fitness > gbest_fitness:
            gbest_fitness = p.pbest_fitness
            gbest_position = np.copy(p.position)

    # Step 6: Iterate Execution Loop
    for iteration in range(max_iterations):
        for p in swarm:
            # Step 4: Evaluate Fitness
            current_fitness = evaluate_fitness(p.position)

            # Update Personal Best
            if current_fitness > p.pbest_fitness:
                p.pbest_fitness = current_fitness
                p.pbest_position = np.copy(p.position)

            # Update Global Best
            if current_fitness > gbest_fitness:
                gbest_fitness = current_fitness
                gbest_position = np.copy(p.position)

        # Step 5: Update Velocities and Positions
        for p in swarm:
            r1 = random.random()
            r2 = random.random()

            cognitive_velocity = c1 * r1 * (p.pbest_position - p.position)
            social_velocity = c2 * r2 * (gbest_position - p.position)

            p.velocity = (w * p.velocity) + cognitive_velocity + social_velocity
            p.update_position()

    # Step 7: Output Best Solution
    print("\n" + "="*55)
    print("      PSO OPTIMAL ROUTINE SPECIFICATIONS GENERATED")
    print("="*55)
    print(f"Cardio Training Volume:   {gbest_position[0]:.1f} hours / week")
    print(f"Resistance/Strength Work: {gbest_position[1]:.1f} hours / week")
    print(f"Mobility & Stretching:    {gbest_position[2]:.1f} hours / week")
    print(f"Daily Caloric Balancing:  {gbest_position[3]:+.0f} kcal / day")
    print(f"Target Rest Allotment:    {int(gbest_position[4])} days / week")
    print("="*55)

if __name__ == "__main__":
    run_pso()


  PSO PERSONALIZED FITNESS ROUTINE OPTIMIZER
Enter user age (e.g., 35): 32
Maximum workout days allowed per week? (1-7): 5

      PSO OPTIMAL ROUTINE SPECIFICATIONS GENERATED
Cardio Training Volume:   3.0 hours / week
Resistance/Strength Work: 5.0 hours / week
Mobility & Stretching:    2.0 hours / week
Daily Caloric Balancing:  -205 kcal / day
Target Rest Allotment:    2 days / week
